# CodeSentinel AI: DevSecOps Analysis Platform
This notebook runs the entire CodeSentinel AI dataset crawling, vector indexing, and model training workflow fully within Google Colab. It features GPU type auto-detection, bitsandbytes quantization configuration, and automated browser downloads.

## 1. Environment Setup
Install the required libraries and verify import path mapping:

In [ ]:
!pip install fastapi uvicorn pydantic pydantic-settings httpx aiohttp beautifulsoup4 trafilatura GitPython pymupdf redis arq anthropic torch transformers accelerate bitsandbytes faiss-cpu structlog prometheus-client pytest pytest-asyncio python-multipart python-dotenv nest-asyncio

import os
import sys

# Ensure current working directory is on the path so 'src' can be imported in Colab
sys.path.append(os.path.abspath('.'))

## 2. Dynamic Hardware Detection and GPU Tuning
Optimizes training variables depending on available VRAM and compute capabilities:

In [ ]:
import torch
from src.models.optimize import detect_gpu_and_configure, get_qlora_config

device_name, opts = detect_gpu_and_configure()
print(f"GPU: {device_name}")
print(f"Mixed Precision Level: {opts['mixed_precision']}")
print(f"Execute torch.compile: {opts['use_compile']}")

## 3. Autonomous Crawler Execution
Trigger async crawlers to fetch vulnerability mapping frameworks from Mitre CWE and OWASP Top 10 web portals:

In [ ]:
import asyncio
from src.crawler.owasp import OwaspCrawler
from src.crawler.mitre import MitreCrawler

async def run_crawlers():
    owasp = OwaspCrawler()
    top10 = await owasp.get_top_10()
    print(f"Retrieved {len(top10)} OWASP vulnerability targets.")
    
    mitre = MitreCrawler()
    cwe_info = await mitre.get_cwe_details(89) # SQL Injection
    print(f"Sample CWE Title: {cwe_info['title'] if cwe_info else 'Unavailable'}")

# Google Colab allows running loops using nested event loops via nest_asyncio
import nest_asyncio
nest_asyncio.apply()
asyncio.run(run_crawlers())

## 4. Fine-Tuning the Code Security Model
Start sequence classification training tasks using the HuggingFace model configurations:

In [ ]:
from src.models.train import ModelTrainer

# Create synthetic examples for execution testing
train_data = [
    {"code_sample": "def query(usr): return db.execute('SELECT * FROM users WHERE username = ' + usr)", "is_vulnerable": 1},
    {"code_sample": "def query(usr): return db.execute('SELECT * FROM users WHERE username = ?', (usr,))", "is_vulnerable": 0},
    {"code_sample": "exec(request.POST['cmd'])", "is_vulnerable": 1},
    {"code_sample": "def add(a, b): return a + b", "is_vulnerable": 0}
] * 10

val_data = [
    {"code_sample": "eval(user_input)", "is_vulnerable": 1},
    {"code_sample": "def log(msg): print(msg)", "is_vulnerable": 0}
] * 4

trainer = ModelTrainer(model_name="microsoft/codebert-base", output_dir="./data/models")
trainer.train(train_data, val_data, epochs=1, batch_size=4)

## 5. Vector Database Compilation
Populate a local FAISS database with code signature logs:

In [ ]:
from src.data.vector_store import SecurityVectorStore

db = SecurityVectorStore(dimension=384)
db.add_samples([
    {"owasp_category": "A03:2021-Injection", "description": "SQL injection flaw", "code_sample": "SELECT * FROM users WHERE name = ' + user"},
    {"owasp_category": "A02:2021-Cryptographic Failures", "description": "Hardcoded md5 cipher usage", "code_sample": "hashlib.md5(passwd.encode())"}
])

db.save_index("./data/vector_index")
print("FAISS vector store saved.")

## 6. Project Export & Automatic Download
Compresses all output source files and triggers automatic browser file downloads directly inside your Colab session:

In [ ]:
import os
import zipfile
from google.colab import files

# Exclude virtualenv, git files, and temporary caches. Keep 'data' to include models and vector indexes.
exclude_dirs = {".venv", ".git", ".pytest_cache", "__pycache__", "logs"}
zip_filename = "CodeSentinelAI_Production.zip"

print(f"Creating complete zip archive: {zip_filename}...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk('.'):
        # Modify dirs in-place to ignore excluded paths
        dirs[:] = [d for d in dirs if d not in exclude_dirs and not d.startswith('.')]
        for file in files_list:
            if file == zip_filename:
                continue
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, '.')
            zipf.write(file_path, arcname)

print("CodeSentinelAI_Production.zip successfully created.")
files.download(zip_filename)